# V1 — the reactive agent

COMS E6998-019 · Lecture 2 · *Design of Production Agentic Systems*

One scanner alert goes in. A patch comes out, or a stated reason why not.

The question to keep asking of every cell below:
**which decisions did the programmer make at design time, and which are
deferred to the model at run time?**

Run this from the `demo/` directory. A live run needs Ollama serving
`qwen3.8` and takes about a minute.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

from lec2_reactive_agent.config import CONFIG
CONFIG.model, CONFIG.budget, CONFIG.fixture.name

## 1. The task is one alert

Intake already ran both scanners over the pinned fixture and normalised
720 findings into one contract: `rule`, `file`, `line`, `message`,
`severity`. The agent's whole input is one of those records.

In [ ]:
from lec2_reactive_agent.cli import select_alert

alert = select_alert(alert_id=CONFIG.worked_example)
alert

## 2. The workspace: a copy, and a boundary

The agent never touches the pinned fixture. It gets a disposable copy,
and `resolve()` is the boundary every tool passes through before it
touches disk.

Two things are withheld from that copy, for different reasons.
`ground-truth/` is outside it entirely. And every *sibling benchmark case*
is moved aside — see §8 for why that matters more than it sounds.

In [ ]:
from lec2_reactive_agent.workspace import AgentWorkspace, resolve
from lec2_reactive_agent.exceptions import PathEscapeException

ws_manager = AgentWorkspace(CONFIG, case='BenchmarkTest00283')
ws = ws_manager.__enter__()   # a notebook cannot hold a `with` across cells

print('cases the agent can see :', len(list(ws.glob('testcode/BenchmarkTest*.py'))))
print('cases withheld          :', len(ws_manager.withheld()))
print('helpers left intact     :', len(list(ws.glob('helpers/*.py'))))

In [ ]:
# The boundary refuses anything outside the workspace root.
for attempt in ['testcode/BenchmarkTest00283.py',
                '../ground-truth/expectedresults-0.1.csv',
                '/etc/passwd']:
    try:
        print('allowed :', resolve(ws, attempt).name)
    except PathEscapeException as exc:
        print('REFUSED :', exc)

## 3. Tools: what the model may ask for

Three, and they are the whole action space. `read_file` and `search` only
read; `edit` is the only one that writes.

`edit` refuses an ambiguous target rather than guessing. A silent
multi-replace produces a patch nobody can explain.

In [ ]:
from lec2_reactive_agent.tools import read_file, search, edit

print(read_file(ws, alert['file'], alert['line'] - 2, alert['line'] + 3))

In [ ]:
try:
    edit(ws, alert['file'], 'import', 'IMPORT')   # appears four times
except ValueError as exc:
    print('REFUSED:', exc)

## 4. The dispatcher owns the observation channel

The model *proposes* a tool call. The runtime executes it, or refuses.

This is the module that makes an agent honest. A model can emit a thought,
an action, **and a fabricated observation** in one response without ever
touching the environment — EnIGMA calls it *soliloquizing*. Because only
the dispatcher writes observations, that failure is structurally impossible
rather than something the prompt has to prevent.

Note that a refusal is an *observation*, not an exception. An exception
ends the run; an observation lets the model read what went wrong and
choose again — which is what makes the loop reactive.

In [ ]:
from lec2_reactive_agent.dispatch import Dispatcher
from lec2_reactive_agent.trace import Trace
import tempfile, pathlib

scratch = pathlib.Path(tempfile.mkdtemp()) / 'demo.jsonl'
trace = Trace('notebook', scratch)
dispatch = Dispatcher(ws, trace, config=CONFIG)

for request in [
    {'name': 'search',    'args': {'pattern': 'get_form_parameter'}},
    {'name': 'rm_rf',     'args': {}},
    {'name': 'read_file', 'args': {'path': '../ground-truth/expectedresults-0.1.csv'}},
    {'name': 'read_file', 'args': {'file': 'x.py'}},          # wrong argument NAME
    {'name': 'read_file', 'args': {'path': alert['file'], 'start': '44'}},  # str, not int
]:
    obs = dispatch(request)
    print(f"{request['name']:<10} ok={str(obs.ok):<6} {obs.result.splitlines()[0][:66]}")

The last two are worth pausing on. Models get argument *names* and
*types* wrong constantly. The dispatcher coerces what is unambiguous
(`'44'` → `44`) and turns the rest into a readable refusal — so a sloppy
call costs a retry, not the run.

## 5. State and its reducers

A node never mutates state. It returns a dict of proposed changes, and
LangGraph merges each key by that field's **reducer**.

| field | reducer | effect |
|---|---|---|
| `messages` | `add_messages` | append |
| `observations` | `operator.add` | append |
| `usage` | `add_usage` | sum the four counters |
| everything else | none | replace |

`usage` having a reducer is the interesting one: a node reports only its
own call's delta, so no node ever holds the running total and none can
clobber it.

In [ ]:
from lec2_reactive_agent.state import AgentState, add_usage
from lec2_reactive_agent.trace import Usage

total = add_usage(Usage(), {'input': 914, 'output': 47})
total = add_usage(total,   {'input': 1452, 'output': 80})
print(total)
print('has a .total attribute?', hasattr(total, 'total'))

There is deliberately **no** total across the four counters. A cache read
costs a fraction of fresh input and output is the expensive one; a single
number cannot answer *did context management help?*

## 6. The graph

Three nodes. `controller` is the only one where a model decides.

Every edge is chosen by a predicate over state — except one.

In [ ]:
from lec2_reactive_agent.graph import build_graph

compiled = build_graph(trace, CONFIG)
print(compiled.get_graph().draw_mermaid())

Read the arrows. Conditional edges render dotted; unconditional ones solid.
**`tools --> controller` is the only solid edge**, and that one line is
what makes this version reactive: after every observation, control returns
to the model and it chooses again.

Remove that edge and you have a fixed workflow that cannot ask a second
question.

## 7. Run it

About a minute. Watch the colours: **cyan** is the model deciding,
**yellow** is its reasoning as it arrives, **white** is the runtime acting.

In [ ]:
trace.terminal('error'); trace.close()       # close the scratch trace from §4
ws_manager.__exit__(None, None, None)        # and release the workspace

from lec2_reactive_agent.reactive_agent import ReactiveAgent
from lec2_reactive_agent.console import Console

console = Console()
console.header('v1-notebook', alert, CONFIG.model, CONFIG.budget)
with ReactiveAgent(alert, CONFIG, console=console) as agent:
    final = agent.execute()
console.report(final, CONFIG.budget, agent.trace_path)

## 8. Did it reason, or did it copy?

OWASP Benchmark ships every weakness **twice**: a vulnerable case and a
safe twin implementing the same endpoint correctly. 758 of its 1,230 cases
are safe ones. So the correct patch for any alert sits in a sibling file,
one `search` away.

That is why §2 withheld them. Turn the withholding off and watch what
changes — this is the same model, the same prompt, the same budget.

In [ ]:
from lec2_reactive_agent.trace import read_events, kinds

events = read_events(agent.trace_path)
searches = [e for e in events if e['kind'] == 'tool_request' and e['tool'] == 'search']
print('status        :', final['status'])
print('model calls   :', final['model_calls'])
print('search calls  :', len(searches))
for e in searches[:6]:
    print('   ', e['args'].get('pattern'))

Measured on the worked alert:

| | siblings present | siblings withheld |
|---|---|---|
| outcome | `accepted` | `budget_exhausted` |
| model calls | 5 | 12 (all of them) |
| patch | correct | none |

And the failure is not ignorance. With the siblings gone the model still
said, in its own words:

> *"This is a SQL injection via an f-string. Fix: use parameterized queries."*

It had the answer and could not turn it into an edit without an example to
copy. Ten searches later it was still looking.

**A trace that looks like investigation can be retrieval.** That is the
question this course keeps asking: what does a trace actually prove?

## 9. The trajectory

The graph shows what the agent *could* do. The trajectory shows what one
run *did*, and who decided each step.

In [ ]:
from lec2_reactive_agent.trajectory import to_text, summarise

print(summarise(events))
print()
print(to_text(events))

## 10. What comes next

| version | adds | who decides |
|---|---|---|
| **V1** | the edge back | the model picks the next action |
| V2 | a planner, and revision when a step fails | the model plans; the runtime retries |
| V3 | a validator between executor and output | an evaluator accepts |
| V4 | reflection and an outer attempt loop | the runtime decides *try again?* |

The graph definition is fixed in all four. Accept, reject-and-retry and
reject-and-stop are predicates over state — **dynamic paths need not change
the graph definition.**